In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option('header',True).load("/Volumes/external-catalog-gcp/external-schema-de-gcp/external-volume-gcp/Employee_Attrition.csv")
display(df)

In [0]:
from pyspark.sql.functions import when, sum, count

# Calculate attrition risk by job satisfaction
risk_df = df.groupBy("JobSatisfaction").agg(
    (sum(when(df.Attrition == "Yes", 1).otherwise(0)) / count("*")).alias("AttritionRisk")
)
display(risk_df)

risk_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("`external-catalog-gcp`.default.employee_attrition")

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog-gcp`.default.employee_attrition")
display(history_df)

In [0]:
df_delta = spark.read.table("`external-catalog-gcp`.`default`.`employee_attrition`")
display(df_delta)

In [0]:
from pyspark.sql import Row

dummy_data = [
    Row(JobSatisfaction="1", AttritionRisk=0.1),
    Row(JobSatisfaction="2", AttritionRisk=0.2),
    Row(JobSatisfaction="3", AttritionRisk=0.3),
    Row(JobSatisfaction="4", AttritionRisk=0.4),
    Row(JobSatisfaction="5", AttritionRisk=0.5)
]

dummy_df = spark.createDataFrame(dummy_data)

dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog-gcp`.default.employee_attrition")

In [0]:
df_delta = spark.read.table("`external-catalog-gcp`.default.employee_attrition")
display(df_delta)

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog-gcp`.default.employee_attrition")
display(history_df)

In [0]:
df_version_1 = spark.read.format("delta").option("versionAsOf", 0).table("`external-catalog-gcp`.default.employee_attrition")
display(df_version_1)

df_version_2 = spark.read.format("delta").option("versionAsOf", 1).table("`external-catalog-gcp`.default.employee_attrition")
display(df_version_2)